# Symbolic Reasoning Agent
**Pipeline:** Natural Language → Rule-based Semantic Parser → Facts + Rules (Horn / FOL clauses / SCM) → Classical Reasoning → True/False

Three classical engines behind one router:
1. **Stratified Forward Chaining + Unification** (closed-world, negation-as-failure) for RuleTaker-style theories
2. **Structural Causal Model + forward propagation** (abduction → intervention → prediction) for CounterBench-style counterfactuals
3. **Resolution Refutation** (CNF, Skolemization, unification, set-of-support, unit-preference best-first search) for FOLIO-style first-order logic

No generative model is used anywhere; every prediction is produced by the engines below.

In [1]:
import re, time, heapq, itertools, json, csv, random
from collections import defaultdict, Counter

def is_var(t):
    return isinstance(t, str) and t.startswith('?')

def fmt_atom(a): return f"{a[0]}({', '.join(x.lstrip('?').upper() if is_var(x) else x for x in a[1])})"
def fmt_lit(l): return ('¬' if l[0] else '') + fmt_atom(l[1])
def fmt_rule(r): return (' ∧ '.join(fmt_lit(l) for l in r[0]) or '⊤') + ' → ' + fmt_lit(r[1])

print("Libraries loaded | is_var('?x') =", is_var('?x'), "| is_var('bob') =", is_var('bob'))

Libraries loaded | is_var('?x') = True | is_var('bob') = False


## 1. Representation
* **Atom** = `(predicate, (arg1, ...))`, e.g. `('young', ('bob',))`, `('eat', ('bear','squirrel'))`
* **Literal** = `(negated: bool, atom)`
* **Rule** = `(body: tuple[Literal], head: Literal)`; variables start with `?`

## 2. Natural Language → Logic parser (deterministic, grammar-based)

In [2]:
VAR_WORDS = {'someone', 'something', 'somebody', 'anyone', 'anything', 'everyone', 'everything',
             'everybody', 'they', 'it', 'them', 'he', 'she', 'him', 'her'}
DETS = {'the', 'a', 'an'}
COPULA = {'is', 'are', 'was', 'were', 'am', 'be'}
MODALS = {'can', 'could', 'will', 'would', 'may', 'might', 'must', 'should'}
GENERIC_HEADS = {'thing', 'things', 'person', 'people', 'persons', 'one', 'ones', 'animal', 'animals',
                 'object', 'objects', 'creature', 'creatures', 'someone', 'something', 'everyone',
                 'everything', 'individual', 'individuals', 'entity', 'entities'}
NO_SING = ('ous', 'ss', 'us', 'is', 'ful', 'less')


def singular(w):
    if w == 'people': return 'person'
    if w.endswith(NO_SING) or len(w) <= 3: return w
    if w.endswith('ies'): return w[:-3] + 'y'
    if w.endswith(('sses', 'xes', 'ches', 'shes', 'zes')): return w[:-2]
    if w.endswith('s'): return w[:-1]
    return w


def lemma(v):
    irregular = {'has': 'have', 'does': 'do', 'goes': 'go'}
    if v in irregular: return irregular[v]
    if v.endswith('ss') or len(v) <= 3: return v
    if v.endswith('ies'): return v[:-3] + 'y'
    if v.endswith(('sses', 'xes', 'ches', 'shes', 'zzes')): return v[:-2]
    if v.endswith('s'): return v[:-1]
    return v


def normalize(s):
    s = s.strip().lower()
    s = re.sub(r'[.!?]+$', '', s).strip()
    for a, b in (("cannot", "can not"), ("can't", "can not"), ("won't", "will not"), ("n't", " not")):
        s = s.replace(a, b)
    s = re.sub(r'\s*,\s*then\b', ' then', s)
    s = re.sub(r'\s*,\s*(?:and\s+)?', ' and ', s)
    s = re.sub(r'[;:"]', ' ', s)
    return re.sub(r'\s+', ' ', s).strip()


def split_sentences(text):
    return [s for s in re.split(r'(?<=[.!?])\s+|\n+', text.strip()) if s.strip()]


for w in ['birds', 'nervous', 'people']: print(f"singular({w}) = {singular(w)}")
for v in ['chases', 'has', 'kisses']: print(f"lemma({v}) = {lemma(v)}")
print("normalize:", normalize("If someone is big, then they aren't red."))

singular(birds) = bird
singular(nervous) = nervous
singular(people) = person
lemma(chases) = chase
lemma(has) = have
lemma(kisses) = kiss
normalize: if someone is big then they are not red


In [3]:
def ent(tokens):
    toks = [t for t in tokens if t]
    if not toks: return None
    if len(toks) == 1 and toks[0] in VAR_WORDS: return '?x'
    if toks[0] in DETS: toks = toks[1:]
    return '_'.join(toks) if toks else None


def split_subject(toks):
    if not toks: return [], []
    if toks[0] in VAR_WORDS: return toks[:1], toks[1:]
    if toks[0] in DETS:
        for i in range(2, len(toks)):
            t = toks[i]
            if t in COPULA or t in MODALS or t in ('does', 'do', 'did') or (t.endswith('s') and not t.endswith('ss')):
                return toks[:i], toks[i:]
        return toks[:2], toks[2:]
    return toks[:1], toks[1:]


for t_ in ["the bald eagle chases the cat", "someone is big", "bob likes the dog"]:
    subj_, rest_ = split_subject(t_.split()); print(f"{t_:<32} subject={subj_}  entity={ent(subj_)}  rest={rest_}")

the bald eagle chases the cat    subject=['the', 'bald', 'eagle']  entity=bald_eagle  rest=['chases', 'the', 'cat']
someone is big                   subject=['someone']  entity=?x  rest=['is', 'big']
bob likes the dog                subject=['bob']  entity=bob  rest=['likes', 'the', 'dog']


In [4]:
def parse_clause(text):
    """One simple clause -> {'subj','kind','aux','lits'} or None."""
    toks = text.split()
    subj, rest = split_subject(toks)
    if not subj or subj[0] in ('not', 'and', 'or') or not rest: return None
    s = ent(subj)
    if s is None: return None
    v = rest[0]
    if v in COPULA:
        neg = rest[1:2] == ['not']
        p = rest[2:] if neg else rest[1:]
        if p and p[0] in DETS: p = p[1:]
        if not p: return None
        if v in ('are', 'were'): p = p[:-1] + [singular(p[-1])]
        return {'subj': subj, 'kind': 'cop', 'aux': v, 'lits': [(neg, ('_'.join(p), (s,)))]}
    if v in MODALS:
        neg = rest[1:2] == ['not']
        vb = rest[2:] if neg else rest[1:]
        if not vb: return None
        return {'subj': subj, 'kind': 'modal', 'aux': v, 'lits': [(neg, (v + '_' + '_'.join(vb), (s,)))]}
    neg = False
    if v in ('does', 'do', 'did') and rest[1:2] == ['not']:
        neg, rest = True, rest[2:]
        if not rest: return None
        v = rest[0]
    vb, obj = lemma(v), rest[1:]
    if obj:
        o = ent(obj)
        if o is None: return None
        return {'subj': subj, 'kind': 'rel', 'aux': None, 'lits': [(neg, (vb, (s, o)))]}
    return {'subj': subj, 'kind': 'rel', 'aux': None, 'lits': [(neg, (vb, (s,)))]}


def parse_conj(text):
    """'someone is big and not red and they eat the cat' -> list of literals (handles elided subjects)."""
    pieces = [p.strip() for p in re.split(r'\band\b', text) if p.strip()]
    lits, prev = [], None
    for p in pieces:
        r = None
        if not p.startswith('not ') and len(p.split()) > 1:
            r = parse_clause(p)
        if r is None and prev is not None:
            subj = ' '.join(prev['subj'])
            aux = (' ' + prev['aux']) if prev['aux'] else ''
            r = parse_clause(f"{subj}{aux} {p}")
        if r is None: return None
        lits += r['lits']
        prev = r
    return lits or None


for c in ["the bald eagle does not chase the cat", "bob is not a bird", "anne can fly"]:
    print(f"{c:<40} -> {[fmt_lit(l) for l in parse_clause(c)['lits']]}")
print("conjunction:", [fmt_lit(l) for l in parse_conj("someone is big and not red and they eat the cat")])

the bald eagle does not chase the cat    -> ['¬chase(bald_eagle, cat)']
bob is not a bird                        -> ['¬bird(bob)']
anne can fly                             -> ['can_fly(anne)']
conjunction: ['big(X)', '¬red(X)', 'eat(X, cat)']


In [5]:
GEN_REL = re.compile(r'^(?:all |every |each |any )?(.+?) (?:that|who|which) (?:is|are) (.+?) (?:is|are) (.+)$')
GEN_ADJ = re.compile(r'^(all |every |each |any )?(.+?) (are|is) (.+)$')


def np_to_body(np):
    """'big and red things' -> [big(?x), red(?x)];  'young bears' -> [young(?x), bear(?x)]"""
    words = [w for w in np.split() if w != 'and']
    if not words: return None
    lits, neg = [], False
    head = words[-1]
    mods = words[:-1]
    if head not in GENERIC_HEADS:
        lits.append((False, (singular(head), ('?x',))))
    for w in mods:
        if w == 'not': neg = True; continue
        if w in DETS: continue
        lits.append((neg, (w, ('?x',)))); neg = False
    return lits


def parse_sentence(sent):
    s = normalize(sent)
    if not s: return None, None
    m = re.match(r'^if (.+?) then (.+)$', s)
    if m:
        body, head = parse_conj(m.group(1)), parse_conj(m.group(2))
        if body is None or head is None: return None, None
        return 'rule', [(tuple(body), h) for h in head]
    m = GEN_REL.match(s)
    if m and m.group(1).split()[-1] in GENERIC_HEADS:
        body = np_to_body(m.group(1))
        extra = parse_conj('someone is ' + m.group(2))
        head = parse_conj('someone is ' + m.group(3))
        if body is not None and extra and head:
            return 'rule', [(tuple(body + extra), h) for h in head]
    m = GEN_ADJ.match(s)
    if m and (m.group(1) or m.group(3) == 'are') and m.group(2).split()[0] not in VAR_WORDS:
        body = np_to_body(m.group(2))
        head = parse_conj('they are ' + m.group(4))
        if body and head:
            return 'rule', [(tuple(body), h) for h in head]
    lits = parse_conj(s)
    if lits: return 'fact', lits
    return None, None


for sent in ["All big, red things are kind.", "Kind people are smart.", "If something eats the squirrel then it is cold.", "Things that are rough are nice."]:
    kind, items = parse_sentence(sent)
    print(f"{sent:<50} -> {kind}: {[fmt_rule(r) for r in items] if kind == 'rule' else [fmt_lit(l) for l in items]}")

All big, red things are kind.                      -> rule: ['big(X) ∧ red(X) → kind(X)']
Kind people are smart.                             -> rule: ['kind(X) → smart(X)']
If something eats the squirrel then it is cold.    -> rule: ['eat(X, squirrel) → cold(X)']
Things that are rough are nice.                    -> rule: ['rough(X) → nice(X)']


In [6]:
_THEORY_CACHE = {}

def parse_theory(context):
    if context in _THEORY_CACHE: return _THEORY_CACHE[context]
    facts, rules, failed = [], [], []
    for sent in split_sentences(context):
        kind, items = parse_sentence(sent)
        if kind == 'fact':
            for l in items:
                if any(is_var(a) for a in l[1][1]): rules.append(((), l))   # "everything is red"
                else: facts.append(l)
        elif kind == 'rule': rules.extend(items)
        else: failed.append(sent)
    out = (tuple(facts), tuple(rules), tuple(failed))
    _THEORY_CACHE[context] = out
    return out


def parse_query(q):
    lits = parse_conj(normalize(q))
    return tuple(lits) if lits else None



demo_theory = "Bob is young. Bob is kind. If someone is young, then they are smart. If someone is smart and kind, then they are successful."
f, r, failed = parse_theory(demo_theory)
print("facts :", [fmt_lit(l) for l in f])
print("rules :", [fmt_rule(x) for x in r])
print("query :", [fmt_lit(l) for l in parse_query("Bob is successful.")], "| unparsed:", list(failed))

facts : ['young(bob)', 'kind(bob)']
rules : ['young(X) → smart(X)', 'smart(X) ∧ kind(X) → successful(X)']
query : ['successful(bob)'] | unparsed: []


## 3. Engine A — Stratified Forward Chaining with Unification
* Rules are grouped into **strata** (a predicate defined using `not p` is evaluated only after `p` is complete) → sound negation-as-failure.
* Body matching = backtracking unification over a predicate index.
* Every derived fact stores its justification → **proof traces**.
* `cwa=True`: closed world (unprovable ⇒ false). `cwa=False`: open world (negations must be derived explicitly).

In [7]:
def subst_atom(atom, th):
    return (atom[0], tuple(th.get(a, a) if is_var(a) else a for a in atom[1]))


def match_atom(pat, fact, th):
    if pat[0] != fact[0] or len(pat[1]) != len(fact[1]): return None
    th2 = th
    for x, y in zip(pat[1], fact[1]):
        if is_var(x):
            if x in th2:
                if th2[x] != y: return None
            else:
                if th2 is th: th2 = dict(th)
                th2[x] = y
        elif x != y: return None
    return th2


class KB:
    def __init__(self):
        self.pos, self.neg = set(), set()
        self.pidx, self.nidx = defaultdict(set), defaultdict(set)
        self.why = {}

    def add(self, n, atom, reason):
        S = self.neg if n else self.pos
        if atom in S: return False
        S.add(atom)
        (self.nidx if n else self.pidx)[atom[0]].add(atom)
        self.why[(n, atom)] = reason
        return True


print("match young(?x) with young(bob):", match_atom(('young', ('?x',)), ('young', ('bob',)), {}))
print("match eat(?x, cat) with eat(bear, dog):", match_atom(('eat', ('?x', 'cat')), ('eat', ('bear', 'dog')), {}))

match young(?x) with young(bob): {'?x': 'bob'}
match eat(?x, cat) with eat(bear, dog): None


In [8]:
def body_matches(body, kb, domain, cwa):
    posl = [l[1] for l in body if not l[0]]
    negl = [l[1] for l in body if l[0]]

    def rec_pos(i, th):
        if i == len(posl):
            yield from (rec_naf(th) if cwa else rec_neg(0, th)); return
        atom = posl[i]
        g = subst_atom(atom, th)
        if not any(is_var(a) for a in g[1]):
            if g in kb.pos: yield from rec_pos(i + 1, th)
            return
        for f in list(kb.pidx.get(atom[0], ())):
            t2 = match_atom(atom, f, th)
            if t2 is not None: yield from rec_pos(i + 1, t2)

    def rec_naf(th):   # closed world: not p holds iff p is not derivable
        free = sorted({a for at in negl for a in at[1] if is_var(a) and a not in th})
        for vals in (itertools.product(domain, repeat=len(free)) if free else [()]):
            t2 = {**th, **dict(zip(free, vals))} if free else th
            if all(subst_atom(at, t2) not in kb.pos for at in negl):
                yield t2

    def rec_neg(j, th):  # open world: not p needs an explicitly derived negation
        if j == len(negl): yield th; return
        atom = negl[j]
        g = subst_atom(atom, th)
        if not any(is_var(a) for a in g[1]):
            if g in kb.neg: yield from rec_neg(j + 1, th)
            return
        for f in list(kb.nidx.get(atom[0], ())):
            t2 = match_atom(atom, f, th)
            if t2 is not None: yield from rec_neg(j + 1, t2)

    yield from rec_pos(0, {})


def stratify(rules):
    pr = [r for r in rules if not r[1][0]]
    st = defaultdict(int)
    limit = len({r[1][1][0] for r in pr}) + 1
    for _ in range(limit + 1):
        ch = False
        for body, (hn, ha) in pr:
            for n, a in body:
                need = st[a[0]] + (1 if n else 0)
                if st[ha[0]] < need: st[ha[0]] = need; ch = True
        if not ch: break
    else:
        return [pr]                       # not stratifiable -> single layer
    if any(v > limit for v in st.values()): return [pr]
    m = max((st[r[1][1][0]] for r in pr), default=0)
    return [[r for r in pr if st[r[1][1][0]] == k] for k in range(m + 1)]


_, demo_rules, _ = parse_theory("If someone is not big then they are red. If someone is red then they are kind. If someone is kind and not young then they are nice.")
for k, layer in enumerate(stratify(demo_rules)):
    if layer: print(f"stratum {k}:", [fmt_rule(x) for x in layer])

stratum 1: ['¬big(X) → red(X)', 'red(X) → kind(X)', 'kind(X) ∧ ¬young(X) → nice(X)']


In [9]:
def forward_chain(facts, rules, domain, cwa=True):
    kb = KB()
    for n, a in facts: kb.add(n, a, ('given',))
    if cwa:
        layers = stratify(rules)
        neg_rules = [r for r in rules if r[1][0]]
        if neg_rules: layers = layers + [neg_rules]
    else:
        layers = [list(rules)]
    for layer in layers:
        changed = True
        while changed:
            changed = False
            for rule in layer:
                body, (hn, ha) = rule
                for th in list(body_matches(body, kb, domain, cwa)):
                    free = sorted({a for a in ha[1] if is_var(a) and a not in th})
                    for vals in (itertools.product(domain, repeat=len(free)) if free else [()]):
                        t2 = {**th, **dict(zip(free, vals))} if free else th
                        if kb.add(hn, subst_atom(ha, t2), ('rule', rule, t2)): changed = True
    return kb


_kb0 = forward_chain(*parse_theory("Anne is red. If someone is not big then they are kind.")[:2], domain=('anne',))
print("closed world  :", sorted(fmt_atom(a) for a in _kb0.pos))
_kb1 = forward_chain(*parse_theory("Anne is red. If someone is not big then they are kind.")[:2], domain=('anne',), cwa=False)
print("open world    :", sorted(fmt_atom(a) for a in _kb1.pos))

closed world  : ['kind(anne)', 'red(anne)']
open world    : ['red(anne)']


In [10]:
def holds(lit, kb, domain, cwa):
    n, a = lit
    vs = sorted({x for x in a[1] if is_var(x)})
    if vs:   # existential query
        return any(holds((n, subst_atom(a, dict(zip(vs, v)))), kb, domain, cwa)
                   for v in itertools.product(domain, repeat=len(vs)))
    if not n: return a in kb.pos
    return a in kb.neg or (cwa and a not in kb.pos)


def constants(facts, rules, query):
    c = set()
    for _, a in facts: c.update(a[1])
    for body, head in rules:
        for _, a in list(body) + [head]: c.update(x for x in a[1] if not is_var(x))
    for _, a in query: c.update(x for x in a[1] if not is_var(x))
    return sorted(c)


_FC_CACHE = {}

def rule_engine(context, statement, cwa=True, return_kb=False):
    facts, rules, failed = parse_theory(context)
    q = parse_query(statement)
    if q is None: return None
    domain = tuple(constants(facts, rules, q))
    key = (context, domain, cwa)
    kb = _FC_CACHE.get(key)
    if kb is None:
        kb = forward_chain(facts, rules, domain, cwa)
        _FC_CACHE[key] = kb
    ans = all(holds(l, kb, domain, cwa) for l in q)
    return (ans, kb, q) if return_kb else ans


demo_kb = forward_chain(*parse_theory(demo_theory)[:2], domain=('bob',))
print("derived facts:", sorted(fmt_atom(a) for a in demo_kb.pos))
for q in ["Bob is successful.", "Bob is not big.", "Bob is big."]:
    print(f"{q:<22} -> {rule_engine(demo_theory, q)}")

derived facts: ['kind(bob)', 'smart(bob)', 'successful(bob)', 'young(bob)']
Bob is successful.     -> True
Bob is not big.        -> True
Bob is big.            -> False


In [11]:
def explain(lit, kb, cwa=True, depth=0, seen=None, done=None):
    """Human-readable proof tree for a literal (sub-proofs already shown are not repeated)."""
    seen = seen or set()
    done = set() if done is None else done
    pad = '    ' * depth
    n, a = lit
    if (n, a) in kb.why and (n, a) not in seen:
        r = kb.why[(n, a)]
        if r[0] == 'given':
            return [f"{pad}{fmt_lit(lit)}   [fact]"]
        if (n, a) in done:
            return [f"{pad}{fmt_lit(lit)}   [proved above]"]
        done.add((n, a))
        seen = seen | {(n, a)}
        rule, th = r[1], r[2]
        out = [f"{pad}{fmt_lit(lit)}   [rule: {fmt_rule(rule)}  θ={{{', '.join(k[1:].upper()+'/'+v for k, v in th.items())}}}]"]
        for bl in rule[0]:
            out += explain((bl[0], subst_atom(bl[1], th)), kb, cwa, depth + 1, seen, done)
        return out
    if n and cwa and a not in kb.pos:
        return [f"{pad}{fmt_lit(lit)}   [negation-as-failure: {fmt_atom(a)} not derivable]"]
    return [f"{pad}{fmt_lit(lit)}   [NOT derivable]"]

ans, kb_, q_ = rule_engine(demo_theory, "Bob is successful.", return_kb=True)
print("\n".join(explain(q_[0], kb_)))

successful(bob)   [rule: smart(X) ∧ kind(X) → successful(X)  θ={X/bob}]
    smart(bob)   [rule: young(X) → smart(X)  θ={X/bob}]
        young(bob)   [fact]
    kind(bob)   [fact]


## 4. Engine B — Structural Causal Model (counterfactuals)
Text → boolean structural equations (`A causes B`, `A and B cause C`, `A or B cause C`, `A causes not B`).
Counterfactual query `Would Y occur if not X instead of X?` is answered by Pearl's 3 steps:
**abduction** (enumerate exogenous worlds consistent with observations/factual premise) →
**action** (do(X=¬x)) → **prediction** (forward propagation in topological order). Answer = True iff Y holds in **every** consistent world.

In [12]:
CAUSE_RE = re.compile(r'^(.+?) (?:directly |jointly |together |can )?(?:causes?|leads? to|results? in|triggers?|produces?) (.+)$')
VERB_RE = re.compile(r'\b(?:causes?|leads? to|results? in|triggers?|produces?)\b')
BERN_RE = re.compile(r'^(.+?)\s*~\s*bern\(\s*([0-9.]+)\s*\)$')
OBS_RE = re.compile(r'^(?:we )?(?:observed?|observe|see|saw|found|notice[d]?|know that)\s+(.+)$')
OBS_POS = re.compile(r'^(.+?) (?:occurred|occurs|happened|happens|is true|was true|is observed|was observed|is present)$')
OBS_NEG = re.compile(r'^(.+?) (?:did not occur|does not occur|did not happen|does not happen|is false|was false|is absent)$')
Q_IF = re.compile(r'would (not )?(.+?) (?:occur|happen|be true|have occurred|be present|be the case)(?: \w+)? if (.+?)(?: instead of (.+?))?$')
Q_NESTED = re.compile(r'\b(?:assume|suppose) ([^.,]+?),? (?:and )?based on (?:this|that) assumption,? (?:further )?(?:suppose|assume) ([^.]+?)\. would (not )?([^.]+?) (?:occur|happen|be true)')
Q_PLAIN = re.compile(r'would (not )?(.+?) (?:occur|happen|be true)$')
# Reading conventions of the counterfactual questions. They are NOT hard-coded: each question type gets the
# configuration that is best on a selection split of the public data (see experiment section).
#   neg_obs / neg_do : 'not Q' in an observation / intervention names the node Q defined by 'P causes not Q'
#   clamp            : observed variables keep their observed value in the counterfactual world
#   incons           : answer when the observations contradict the factual world
#   multi            : 'all' interventions, or only the 'first' one
SCM_DEFAULT = {'neg_obs': False, 'neg_do': False, 'clamp': False, 'incons': False, 'multi': 'all'}
SCM_CFG = {}                 # question type -> config  (filled by the experiment)


def cvar(s):
    s = re.sub(r'^(?:the|a|an) ', '', s.strip().lower())
    return re.sub(r'\s+', ' ', s).strip()


def lit_list(s):
    """'Razz, Pex and not Zurn' -> [('razz',False), ('pex',False), ('zurn',True)]"""
    out = []
    for x in re.split(r',|\band\b|\bor\b', re.sub(r'\b(?:either|both|together|jointly)\b', ' ', s)):
        x = x.strip()
        if not x: continue
        neg = x.startswith('not ')
        v = cvar(x[4:] if neg else x)
        if v: out.append((v, neg))
    return out


def causal_fragments(sent):
    """Split 'A causes B, C and D, not D causes E, A and B causes F, and E or B causes G' into causal clauses."""
    parts = []
    for p in re.split(r',\s*(?:and\s+)?', sent):
        p = p.strip()
        while True:   # 'A causes B and C causes D' -> split only if the left side already has a verb
            m = re.search(r'\s+and\s+(?=(?:not\s+)?\w+\s+(?:causes?|leads?)\b)', p)
            if m and VERB_RE.search(p[:m.start()]):
                parts.append(p[:m.start()]); p = p[m.end():]
            else: break
        if p: parts.append(p)
    frags = []
    for k, p in enumerate(parts):
        if VERB_RE.search(p):
            frags.append(p)
        elif k + 1 < len(parts) and (not frags or re.match(r'^(?:not\s+)?\w+\s+(?:and|or)\s+', parts[k + 1])) \
                and VERB_RE.search(parts[k + 1]) and (not frags or ' together ' in ' ' + parts[k + 1] + ' '):
            parts[k + 1] = p + ' and ' + parts[k + 1]      # first member of a joint cause list
        elif frags:
            frags[-1] += ' and ' + p                       # extra effect of the previous clause
    return frags


demo_cb = "We know that Glent causes Razz, Pex and Zurn, not Zurn causes Melf, and Melf and Razz together cause Zlim."
for fr in causal_fragments(demo_cb.lower().replace("we know that ", "")):
    print("clause:", fr)

clause: glent causes razz and pex and zurn
clause: not zurn causes melf
clause: melf and razz together cause zlim.


In [13]:
def parse_causal(text):
    """-> eqs {effect: [(op, [(cause, neg)], out_neg)]}, priors {root: p}, evidence list, query dict"""
    eqs, priors, evidence, query = defaultdict(list), {}, [], None
    t = text.lower().replace('?', '.').replace('\n', ' ')
    t = re.sub(r'\s+(?=(?:we know that|we observed?|we see|we saw)\b)', '. ', t)   # missing full stops
    m = Q_NESTED.search(t)
    if m:
        query = {'neg': bool(m.group(3)), 'target': cvar(m.group(4)),
                 'do': [m.group(1), m.group(2)], 'factual': None}
        t = t[:m.start()] + t[m.end():]
    for sent in re.split(r'(?<!\d)[.;](?!\d)', t):
        sent = re.sub(r'\s+', ' ', sent).strip(' ,')
        if not sent or 'direct effect' in sent or sent.startswith('imagine'): continue
        sent = re.sub(r'^(?:and )?(?:we know that|it is known that|suppose that|assume that|given that)\s+', '', sent)
        if query is None:
            qm = Q_IF.search(sent)
            if qm:
                query = {'neg': bool(qm.group(1)), 'target': cvar(qm.group(2)), 'do': [qm.group(3)], 'factual': qm.group(4)}
                continue
            qm = Q_PLAIN.search(sent)
            if qm and sent.startswith('would'):
                query = {'neg': bool(qm.group(1)), 'target': cvar(qm.group(2)), 'do': [], 'factual': None}; continue
        b = BERN_RE.match(sent)
        if b: priors[cvar(b.group(1))] = float(b.group(2)); continue
        if VERB_RE.search(sent):
            for f in causal_fragments(sent):
                cm = CAUSE_RE.match(f)
                if not cm: continue
                lhs, rhs = cm.group(1), cm.group(2)
                op = 'or' if re.search(r'\bor\b|\beither\b', lhs) else 'and'
                causes = lit_list(lhs)
                for eff, eneg in lit_list(rhs):
                    eqs[eff].append((op, causes, eneg))
            continue
        om = OBS_RE.match(sent)
        body = om.group(1) if om else sent
        m2 = OBS_NEG.match(body)
        if m2: evidence.append(m2.group(1) + ' :neg'); continue
        m2 = OBS_POS.match(body)
        if m2: evidence.append(m2.group(1)); continue
        if om: evidence.append(body)
    return eqs, priors, evidence, query


def to_assign(s, negvars, neg_mode=False):
    out = {}
    if not s: return out
    neg_all = s.endswith(' :neg')
    for v, neg in lit_list(s.replace(' :neg', '')):
        val = not neg
        if neg_all: val = not val
        if neg_mode and v in negvars: val = not val     # 'not Q' names the negated node itself
        out[v] = val
    return out


print(to_assign("not glent and pex", set()), "| observation:", to_assign("flen :neg", set()))

{'glent': False, 'pex': True} | observation: {'flen': False}


In [14]:
def scm_eval(eqs, roots, do):
    memo = {}
    def val(v, stack=()):
        if v in do: return do[v]
        if v in memo: return memo[v]
        if v not in eqs: return roots.get(v, False)
        if v in stack: return False
        terms = []
        for op, causes, oneg in eqs[v]:
            vals = [val(c, stack + (v,)) != cn for c, cn in causes]
            r = all(vals) if op == 'and' else any(vals)
            terms.append((not r) if oneg else r)
        memo[v] = any(terms)
        return memo[v]
    return val


eqs_, pri_, ev_, q_ = parse_causal(demo_cb + " Would Zlim occur if not Glent instead of Glent?")
for v, ts in eqs_.items(): print(f"  {v} <- {ts}")
print("query:", q_)

  razz <- [('and', [('glent', False)], False)]
  pex <- [('and', [('glent', False)], False)]
  zurn <- [('and', [('glent', False)], False)]
  melf <- [('and', [('zurn', True)], False)]
  zlim <- [('and', [('melf', False), ('razz', False)], False)]
query: {'neg': False, 'target': 'zlim', 'do': ['not glent'], 'factual': 'glent'}


In [15]:
def question_type(q, evidence, priors):
    """Detect the counterfactual question type from the text alone (no dataset labels are used)."""
    if len(q['do']) > 1: return 'nested'
    if q['factual'] is None and len(lit_list(q['do'][0] if q['do'] else '')) > 1: return 'joint'
    if priors: return 'backdoor'
    if evidence: return 'conditional'
    return 'basic'


def causal_engine(context, statement, return_info=False, cfg=None):
    """Pearl's 3-step counterfactual: abduction (posterior over exogenous worlds, Bernoulli priors) ->
    action (do) -> prediction (forward propagation). Answer = True iff P(target | evidence, do) > 0.5."""
    eqs, priors, evidence, q = parse_causal(context + '. ' + statement)
    if q is None or not eqs: return None
    qtype = question_type(q, evidence, priors)
    cfg = cfg or SCM_CFG.get(qtype, SCM_DEFAULT)
    negvars = {e for e, ts in eqs.items() if any(o for _, _, o in ts)}
    do = {}
    for d in q['do']: do.update(to_assign(d, negvars, cfg['neg_do']))     # nested: later assumption overrides
    if cfg['multi'] == 'first' and len(do) > 1:
        k = next(iter(do)); do = {k: do[k]}
    if q['factual'] is not None: factual = to_assign(q['factual'], negvars, cfg['neg_do'])
    elif len(q['do']) == 1: factual = {v: (not b) for v, b in do.items() if v not in eqs}   # 'if not X' presupposes X
    else: factual = {}
    obs = {}
    for e in evidence: obs.update(to_assign(e, negvars, cfg['neg_obs']))
    cf_do = {**obs, **do} if cfg['clamp'] else do
    allvars = set(eqs) | {c for ts in eqs.values() for _, cs, _ in ts for c, _ in cs} | set(obs) | set(do) | {q['target']}
    roots = sorted(v for v in allvars if v not in eqs)
    constraints = {**factual, **obs}
    num = den = 0.0
    for bits in itertools.product([True, False], repeat=min(len(roots), 16)):
        w = dict(zip(roots, bits))
        val = scm_eval(eqs, w, {})
        if not all(val(v) == b for v, b in constraints.items()): continue
        p = 1.0
        for r, b in w.items(): p *= priors.get(r, 0.5) if b else 1 - priors.get(r, 0.5)
        den += p
        num += p * scm_eval(eqs, w, cf_do)(q['target'])
    ans = cfg['incons'] if den == 0 else (num / den > 0.5)
    ans = (not ans) if q['neg'] else ans
    info = {'eqs': dict(eqs), 'do': cf_do, 'obs': obs, 'factual': factual, 'roots': roots, 'type': qtype}
    return (ans, info) if return_info else ans


for qq in ["Would Zlim occur if not Glent instead of Glent?", "Would Zlim occur if Glent instead of not Glent?"]:
    a_, info_ = causal_engine(demo_cb, qq, return_info=True)
    print(f"{qq:<50} -> {a_}   (type: {info_['type']})")

Would Zlim occur if not Glent instead of Glent?    -> False   (type: basic)
Would Zlim occur if Glent instead of not Glent?    -> False   (type: basic)


In [16]:
SCM_GRID = [dict(zip(SCM_DEFAULT, v)) for v in itertools.product([False, True], [False, True], [False, True], [False, True], ['all', 'first'])]


def tune_scm(examples, seed=0):
    """Choose the reading convention per question type on a 50% selection split; report on the held-out 50%."""
    global SCM_CFG
    rng = random.Random(seed); idx = list(range(len(examples))); rng.shuffle(idx)
    sel, held = [examples[i] for i in idx[::2]], [examples[i] for i in idx[1::2]]
    def qt(ex):
        r = causal_engine(ex['context'], ex['statement'], return_info=True, cfg=SCM_DEFAULT)
        return r[1]['type'] if r else None
    groups = defaultdict(list)
    for ex in sel: groups[qt(ex)].append(ex)
    SCM_CFG = {}
    for t, exs in groups.items():
        if t is None: continue
        scores = [(sum(causal_engine(e['context'], e['statement'], cfg=c) == e['label'] for e in exs), -k, c) for k, c in enumerate(SCM_GRID)]
        best, _, cfg = max(scores, key=lambda x: (x[0], x[1]))
        base = sum(causal_engine(e['context'], e['statement'], cfg=SCM_DEFAULT) == e['label'] for e in exs)
        margin = max(3, 0.02 * len(exs))          # switch only on a clear gain (guards against noise)
        if best - base < margin: cfg, best = SCM_DEFAULT, base
        SCM_CFG[t] = cfg
        tag = 'default kept' if cfg is SCM_DEFAULT else 'switched'
        print(f"  {t:>12}: default {100*base/len(exs):6.2f}%  ->  chosen {100*best/len(exs):6.2f}%  ({tag})  cfg={cfg}")
    return sel, held


def fmt_scm(info):
    lines = []
    for v, ts in info['eqs'].items():
        rhs = ' ∨ '.join(('¬' if o else '') + '(' + (' ∧ ' if op == 'and' else ' ∨ ').join(('¬' if n else '') + c for c, n in cs) + ')' for op, cs, o in ts)
        lines.append(f"  {v} := {rhs}")
    lines.append(f"  type: {info.get('type')}   evidence/factual: {info['factual'] | info['obs']}   do: {info['do']}")
    return '\n'.join(lines)

print("search space per question type:", len(SCM_GRID), "reading conventions")
print(fmt_scm(causal_engine(demo_cb, "Would Zlim occur if not Glent instead of Glent?", return_info=True)[1]))

search space per question type: 32 reading conventions
  razz := (glent)
  pex := (glent)
  zurn := (glent)
  melf := (¬zurn)
  zlim := (melf ∧ razz)
  type: basic   evidence/factual: {'glent': True}   do: {'glent': False}


## 5. Engine C — First-Order Logic Resolution Prover
FOL string → AST (∀ ∃ ¬ ∧ ∨ → ↔ ⊕, n-ary predicates, functions) → eliminate →,↔,⊕ → NNF → standardize apart + **Skolemize** → CNF.
Query: KB ∧ ¬Q is refuted with **binary resolution + factoring + unification (occurs check)**,
**set-of-support** strategy and **best-first (shortest clause first, i.e. unit preference)** search. Empty clause ⇒ entailed (True).

In [17]:
FOL_TOK = re.compile(r'\s*(∀|∃|¬|∧|∨|→|↔|⊕|\(|\)|,|≠|=|[^\s∀∃¬∧∨→↔⊕(),=≠]+)')


def normalize_logic(s):
    """Accept common ASCII / textbook / Prolog-style logic syntaxes and map them to one notation."""
    s = s.strip().rstrip('.')
    if ':-' in s:                                   # Prolog rule  head :- b1, b2.
        head, body = s.split(':-', 1)
        body = re.sub(r'\\\+\s*', '¬', body)
        body = ' ∧ '.join(b.strip() for b in re.split(r',\s*(?![^()]*\))', body))
        vs = sorted(set(re.findall(r'\b([A-Z]\w*)\b(?=\s*[,)])', head + body)))
        for v in vs:
            head = re.sub(rf'\b{v}\b(?=\s*[,)])', v.lower(), head); body = re.sub(rf'\b{v}\b(?=\s*[,)])', v.lower(), body)
        s = ''.join(f'∀{v.lower()} ' for v in vs) + f'(({body}) → {head.strip()})'
    for a, b in (('<=>', '↔'), ('<->', '↔'), ('=>', '→'), ('->', '→'), ('&&', '∧'), ('||', '∨'), ('⇒', '→'), ('⇔', '↔'), ('⟷', '↔'), ('∼', '¬'), ('!', '¬')):
        s = s.replace(a, b)
    s = re.sub(r'\b(?:forall|all|FORALL)\s+(\w+)\s*[.:]?', r'∀\1 ', s)
    s = re.sub(r'\b(?:exists|EXISTS)\s+(\w+)\s*[.:]?', r'∃\1 ', s)
    s = re.sub(r'([∀∃])\s*(\w+)\s*[.:]', r'\1\2 ', s)
    s = re.sub(r'\bnot\s+', '¬', s); s = re.sub(r'\s+and\s+', ' ∧ ', s); s = re.sub(r'\s+or\s+', ' ∨ ', s)
    s = re.sub(r'\s+xor\s+', ' ⊕ ', s); s = re.sub(r'\s+implies\s+', ' → ', s)
    return s


def fol_tokens(s):
    s = normalize_logic(s)
    s = (s.replace('<->', '↔').replace('->', '→').replace('&', '∧').replace('|', '∨').replace('~', '¬')
          .replace('⟷', '↔').replace('⇒', '→').replace('⇔', '↔').replace('∨¬', '∨ ¬'))
    s = re.sub(r'\bforall\s+', '∀', s); s = re.sub(r'\bexists\s+', '∃', s)
    return [t for t in FOL_TOK.findall(s) if t.strip()]


for f_ in ["forall x. (Dog(x) -> Mammal(x))", "smart(X) :- young(X), kind(X).", "∀x (Tall(x) ⊕ Short(x))"]:
    print(f"{f_:<36} -> {' '.join(fol_tokens(f_))}")

forall x. (Dog(x) -> Mammal(x))      -> ∀ x ( Dog ( x ) → Mammal ( x ) )
smart(X) :- young(X), kind(X).       -> ∀ x ( ( young ( x ) ∧ kind ( x ) ) → smart ( x ) )
∀x (Tall(x) ⊕ Short(x))              -> ∀ x ( Tall ( x ) ⊕ Short ( x ) )


In [18]:
class FOLParser:
    def __init__(self, s): self.t, self.i = fol_tokens(s), 0
    def peek(self): return self.t[self.i] if self.i < len(self.t) else None
    def eat(self, x=None):
        t = self.peek()
        if t is None or (x is not None and t != x): raise ValueError(f"expected {x} got {t}")
        self.i += 1; return t
    def parse(self):
        f = self.iff(frozenset())
        if self.peek() is not None: raise ValueError('trailing tokens: ' + ' '.join(self.t[self.i:]))
        return f
    def iff(self, b):
        l = self.imp(b)
        while self.peek() == '↔': self.eat(); l = ('iff', l, self.imp(b))
        return l
    def imp(self, b):
        l = self.disj(b)
        if self.peek() == '→': self.eat(); return ('imp', l, self.imp(b))
        return l
    def disj(self, b):
        l = self.conj(b)
        while self.peek() in ('∨', '⊕'):
            op = 'or' if self.eat() == '∨' else 'xor'
            l = (op, l, self.conj(b))
        return l
    def conj(self, b):
        l = self.unary(b)
        while self.peek() == '∧': self.eat(); l = ('and', l, self.unary(b))
        return l
    def unary(self, b):
        t = self.peek()
        if t == '¬': self.eat(); return ('not', self.unary(b))
        if t in ('∀', '∃'):
            self.eat(); v = self.eat()
            return ('all' if t == '∀' else 'ex', v, self.iff(b | {v}))   # scope extends right
        if t == '(':
            self.eat(); f = self.iff(b); self.eat(')'); return f
        return self.atom(b)
    def term(self, b):
        name = self.eat()
        if self.peek() == '(':
            self.eat(); args = [self.term(b)]
            while self.peek() == ',': self.eat(); args.append(self.term(b))
            self.eat(')')
            return (name.lower(),) + tuple(args)
        if name in b or re.fullmatch(r'[a-z]', name): return '?' + name
        return name.lower()
    def atom(self, b):
        name = self.eat()
        args = ()
        if self.peek() == '(':
            self.eat(); a = [self.term(b)]
            while self.peek() == ',': self.eat(); a.append(self.term(b))
            self.eat(')'); args = tuple(a)
        if self.peek() in ('=', '≠'):
            op = self.eat()
            lhs = (name.lower(),) + args if args else ('?' + name if (name in b or re.fullmatch(r'[a-z]', name)) else name.lower())
            f = ('pred', '=', (lhs, self.term(b)))
            return ('not', f) if op == '≠' else f
        return ('pred', name.lower(), args)


print(FOLParser("∀x (Dog(x) ∧ ¬Wild(x) → Pet(x))").parse())

('all', 'x', ('imp', ('and', ('pred', 'dog', ('?x',)), ('not', ('pred', 'wild', ('?x',)))), ('pred', 'pet', ('?x',))))


In [19]:
def elim(f):
    op = f[0]
    if op == 'pred': return f
    if op == 'not': return ('not', elim(f[1]))
    if op in ('and', 'or'): return (op, elim(f[1]), elim(f[2]))
    if op == 'imp': return ('or', ('not', elim(f[1])), elim(f[2]))
    a, b = (elim(f[1]), elim(f[2])) if op in ('iff', 'xor') else (None, None)
    if op == 'iff': return ('and', ('or', ('not', a), b), ('or', a, ('not', b)))
    if op == 'xor': return ('and', ('or', a, b), ('or', ('not', a), ('not', b)))
    return (op, f[1], elim(f[2]))


def nnf(f, neg=False):
    op = f[0]
    if op == 'pred': return ('not', f) if neg else f
    if op == 'not': return nnf(f[1], not neg)
    if op in ('and', 'or'):
        nop = {'and': 'or', 'or': 'and'}[op] if neg else op
        return (nop, nnf(f[1], neg), nnf(f[2], neg))
    nop = {'all': 'ex', 'ex': 'all'}[op] if neg else op
    return (nop, f[1], nnf(f[2], neg))


print("NNF of ¬(A ∧ B):", nnf(elim(FOLParser("¬(A(x) ∧ B(x))").parse())))

NNF of ¬(A ∧ B): ('or', ('not', ('pred', 'a', ('?x',))), ('not', ('pred', 'b', ('?x',))))


In [20]:
def sub_term(t, env):
    if isinstance(t, str): return env.get(t, t)
    return (t[0],) + tuple(sub_term(a, env) for a in t[1:])


def skolemize(f, env, univ, ctr):
    op = f[0]
    if op == 'pred': return ('pred', f[1], tuple(sub_term(a, env) for a in f[2]))
    if op == 'not': return ('not', skolemize(f[1], env, univ, ctr))
    if op in ('and', 'or'): return (op, skolemize(f[1], env, univ, ctr), skolemize(f[2], env, univ, ctr))
    v = '?' + f[1]
    if op == 'all':
        nv = f'?{f[1]}_{next(ctr)}'
        return skolemize(f[2], {**env, v: nv}, univ + [nv], ctr)
    sk = f'$sk{next(ctr)}'
    return skolemize(f[2], {**env, v: ((sk,) + tuple(univ)) if univ else sk}, univ, ctr)


def to_clauses(f, cap=4000):
    op = f[0]
    if op == 'and': return to_clauses(f[1], cap) + to_clauses(f[2], cap)
    if op == 'or':
        A, B = to_clauses(f[1], cap), to_clauses(f[2], cap)
        if len(A) * len(B) > cap: raise ValueError('CNF blow-up')
        return [a | b for a in A for b in B]
    if op == 'not': return [frozenset([(False, f[1][1], f[1][2])])]
    return [frozenset([(True, f[1], f[2])])]


def fmt_clause(c):
    if not c: return '□ (empty clause)'
    def ft(t):
        if isinstance(t, tuple): return f"{t[0]}({', '.join(ft(a) for a in t[1:])})"
        return t.lstrip('?').upper() if is_var(t) else t
    return ' ∨ '.join(('' if s else '¬') + f"{p}({', '.join(ft(a) for a in args)})" for s, p, args in sorted(c, key=str))


def is_taut(c):
    return any((not s, p, a) in c for s, p, a in c)


def fol_to_cnf(formula_ast, ctr):
    f = skolemize(nnf(elim(formula_ast)), {}, [], ctr)
    return [c for c in to_clauses(f) if not is_taut(c)]

# --- unification ---
for f_ in ["∀x (Dog(x) → ∃y (Owner(y, x)))", "Tall(max) ⊕ Short(max)"]:
    print(f"{f_:<32} -> CNF clauses: {[fmt_clause(c) for c in fol_to_cnf(FOLParser(f_).parse(), itertools.count())]}")

∀x (Dog(x) → ∃y (Owner(y, x)))   -> CNF clauses: ['¬dog(X_0) ∨ owner($sk1(X_0), X_0)']
Tall(max) ⊕ Short(max)           -> CNF clauses: ['short(max) ∨ tall(max)', '¬short(max) ∨ ¬tall(max)']


In [21]:
def walk(t, s):
    while isinstance(t, str) and t.startswith('?') and t in s: t = s[t]
    return t

def occurs(v, t, s):
    t = walk(t, s)
    if t == v: return True
    return isinstance(t, tuple) and any(occurs(v, a, s) for a in t[1:])

def unify(a, b, s):
    a, b = walk(a, s), walk(b, s)
    if a == b: return s
    if is_var(a): return None if occurs(a, b, s) else {**s, a: b}
    if is_var(b): return None if occurs(b, a, s) else {**s, b: a}
    if isinstance(a, tuple) and isinstance(b, tuple) and a[0] == b[0] and len(a) == len(b):
        for x, y in zip(a[1:], b[1:]):
            s = unify(x, y, s)
            if s is None: return None
        return s
    return None

def unify_args(xs, ys, s):
    if len(xs) != len(ys): return None
    for x, y in zip(xs, ys):
        s = unify(x, y, s)
        if s is None: return None
    return s

def apply_t(t, s):
    t = walk(t, s)
    if isinstance(t, tuple): return (t[0],) + tuple(apply_t(a, s) for a in t[1:])
    return t

def apply_lit(l, s): return (l[0], l[1], tuple(apply_t(a, s) for a in l[2]))

def depth(t): return 1 + max((depth(a) for a in t[1:]), default=0) if isinstance(t, tuple) else 0

print("unify Likes(?x, bob) / Likes(ann, ?y):", unify_args(('?x', 'bob'), ('ann', '?y'), {}))
print("occurs check ?x / f(?x):", unify('?x', ('f', '?x'), {}))

unify Likes(?x, bob) / Likes(ann, ?y): {'?x': 'ann', '?y': 'bob'}
occurs check ?x / f(?x): None


In [22]:
def canon(clause):
    """Rename variables canonically so duplicate clauses are detected."""
    lits = sorted(clause, key=lambda l: (l[1], l[0], repr(re.sub(r"\?[\w$]+", '?', repr(l[2])))))
    mp = {}
    def r(t):
        if is_var(t):
            if t not in mp: mp[t] = f'?v{len(mp)}'
            return mp[t]
        if isinstance(t, tuple): return (t[0],) + tuple(r(a) for a in t[1:])
        return t
    return frozenset((s, p, tuple(r(a) for a in args)) for s, p, args in lits)

def rename_apart(c, tag):
    def r(t):
        if is_var(t): return t + tag
        if isinstance(t, tuple): return (t[0],) + tuple(r(a) for a in t[1:])
        return t
    return frozenset((s, p, tuple(r(a) for a in args)) for s, p, args in c)


def one_way(pat, tgt, s):
    """Match pattern args onto target args, binding only pattern variables (for subsumption)."""
    for x, y in zip(pat, tgt):
        if is_var(x):
            if x in s:
                if s[x] != y: return None
            else: s = {**s, x: y}
        elif isinstance(x, tuple):
            if not (isinstance(y, tuple) and x[0] == y[0] and len(x) == len(y)): return None
            s = one_way(x[1:], y[1:], s)
            if s is None: return None
        elif x != y: return None
    return s


print("canonical:", canon(frozenset({(True, 'p', ('?a', '?b')), (False, 'q', ('?b',))})))
print("subsumes p(?x) -> p(bob):", one_way(('?x',), ('bob',), {}))

canonical: frozenset({(True, 'p', ('?v0', '?v1')), (False, 'q', ('?v1',))})
subsumes p(?x) -> p(bob): {'?x': 'bob'}


In [23]:
def resolution_prove(kb_clauses, goal_clauses, timeout=1.0, max_lits=6, max_depth=3, max_clauses=4000):
    """Set-of-support, best-first (shortest clause first) resolution refutation.
    Returns (True|False|None, proof_parents)  None = gave up (timeout)."""
    t0 = time.perf_counter()
    seen, parents = set(), {}
    usable_idx = defaultdict(list)      # (sign, pred) -> clauses
    units = defaultdict(list)           # (sign, pred) -> unit literal args (for forward subsumption)

    def subsumed(c):
        for s, p, args in c:
            for uargs in units.get((s, p), ()):
                if len(uargs) == len(args) and one_way(uargs, args, {}) is not None: return True
        return False
    heap, tick = [], itertools.count()

    def add_usable(c):
        for s, p, _ in c: usable_idx[(s, p)].append(c)

    for c in kb_clauses:
        c = canon(c)
        if c not in seen:
            seen.add(c); add_usable(c); parents[c] = ('premise',)
            if len(c) == 1: (l,) = c; units[(l[0], l[1])].append(l[2])
    for c in goal_clauses:
        c = canon(c)
        if not c: return True, parents
        if c not in seen: seen.add(c); parents[c] = ('negated goal',); heapq.heappush(heap, (len(c), next(tick), c))

    while heap:
        if time.perf_counter() - t0 > timeout or len(seen) > max_clauses: return None, parents
        _, _, g = heapq.heappop(heap)
        g2 = rename_apart(g, '_g')
        partners = {}
        for s, p, _ in g:
            for u in usable_idx.get((not s, p), ()): partners[id(u)] = u
        partners[id(g)] = g
        new = []
        # factoring
        gl = list(g2)
        for i in range(len(gl)):
            for j in range(i + 1, len(gl)):
                if gl[i][0] == gl[j][0] and gl[i][1] == gl[j][1]:
                    th = unify_args(gl[i][2], gl[j][2], {})
                    if th is not None: new.append((frozenset(apply_lit(l, th) for l in g2 if l != gl[j]), g, g))
        # binary resolution
        for u in partners.values():
            u2 = rename_apart(u, '_u')
            for l1 in g2:
                for l2 in u2:
                    if l1[0] != l2[0] and l1[1] == l2[1]:
                        th = unify_args(l1[2], l2[2], {})
                        if th is not None:
                            r = frozenset(apply_lit(l, th) for l in (g2 - {l1}) | (u2 - {l2}))
                            new.append((r, g, u))
        for r, pa, pb in new:
            if not r:
                parents[frozenset()] = ('resolve', pa, pb); return True, parents
            if len(r) > max_lits or is_taut(r): continue
            if any(depth(a) > max_depth for _, _, args in r for a in args): continue
            rc = canon(r)
            if rc in seen or subsumed(rc): continue
            seen.add(rc); parents[rc] = ('resolve', pa, pb)
            if len(rc) == 1: (l,) = rc; units[(l[0], l[1])].append(l[2])
            heapq.heappush(heap, (len(rc), next(tick), rc))
        add_usable(g)
    return False, parents


demo_kb_cnf = [c for p in ["∀x (Dog(x) → Mammal(x))", "Dog(max)"] for c in fol_to_cnf(FOLParser(p).parse(), itertools.count())]
demo_goal = fol_to_cnf(('not', FOLParser("Mammal(max)").parse()), itertools.count(100))
print("KB ∧ ¬goal refuted (goal entailed):", resolution_prove(demo_kb_cnf, demo_goal)[0])

KB ∧ ¬goal refuted (goal entailed): True


In [24]:
def proof_lines(parents, c=frozenset(), seen=None, out=None):
    seen = set() if seen is None else seen; out = [] if out is None else out
    if c in seen: return out
    seen.add(c)
    info = parents.get(c, ('?',))
    if info[0] == 'resolve':
        proof_lines(parents, info[1], seen, out); proof_lines(parents, info[2], seen, out)
        out.append(f"  {fmt_clause(c)}    ⇐ resolve[{fmt_clause(info[1])}] + [{fmt_clause(info[2])}]")
    else:
        out.append(f"  {fmt_clause(c)}    [{info[0]}]")
    return out


_, demo_par = resolution_prove(demo_kb_cnf, demo_goal)
print("\n".join(proof_lines(demo_par)))

  ¬mammal(max)    [negated goal]
  ¬dog(V0) ∨ mammal(V0)    [premise]
  ¬dog(max)    ⇐ resolve[¬mammal(max)] + [¬dog(V0) ∨ mammal(V0)]
  dog(max)    [premise]
  □ (empty clause)    ⇐ resolve[¬dog(max)] + [dog(max)]


In [25]:
def parse_fol_robust(s):
    """Parse an FOL string; repair common annotation noise (unbalanced brackets, ',' used as ∧)."""
    tries = [s]
    d = s.count('(') - s.count(')')
    if d < 0: tries += ['(' * -d + s, s.rstrip()[:d]]
    if d > 0: tries.append(s + ')' * d)
    tries += [re.sub(r'\)\s*,\s*(?=¬?\w+\()', ') ∧ ', t) for t in list(tries)]
    for t in tries:
        try: return FOLParser(t).parse()
        except Exception: pass
    raise ValueError('unparsable FOL: ' + s)


print(parse_fol_robust("(Spill(peter) ∧ OnlyChild(peter)) ∨ ¬Spill(peter) ∧ ¬OnlyChild(peter))"))

('or', ('and', ('pred', 'spill', ('peter',)), ('pred', 'onlychild', ('peter',))), ('and', ('not', ('pred', 'spill', ('peter',))), ('not', ('pred', 'onlychild', ('peter',)))))


In [26]:
def split_formulas(text):
    if isinstance(text, (list, tuple)): return [t for t in text if str(t).strip()]
    parts = re.split(r'\n+|;|(?<=[)\w])\.\s+', text)
    return [p.strip() for p in parts if p.strip()]


def _lit(f):
    neg = False
    while f[0] == 'not': neg, f = not neg, f[1]
    return (neg, (f[1], f[2])) if f[0] == 'pred' else None


def _conj(f):
    if f[0] == 'and':
        a, b = _conj(f[1]), _conj(f[2])
        return a + b if a is not None and b is not None else None
    l = _lit(f)
    return [l] if l else None


def horn_rule(f):
    """∀x (L1 ∧ ... ∧ Ln → L) or a literal  ->  (body, head) for the forward chainer, else None."""
    while f[0] == 'all': f = f[2]
    l = _lit(f)
    if l: return ((), l)
    if f[0] == 'imp':
        body, head = _conj(f[1]), _lit(f[2])
        if body is not None and head is not None: return (tuple(body), head)
    return None


def logic_engine(premises, conclusion, cwa=None, return_route=False):
    """Logic given directly. Horn-style theories (facts + 'if conjunction then literal' rules, like RuleTaker)
    -> stratified forward chaining with negation-as-failure; anything richer (∨, ⊕, ↔, ∃, nested ∀) -> resolution."""
    prem = split_formulas(premises)
    try:
        asts = [parse_fol_robust(p) for p in prem]
        goal = parse_fol_robust(conclusion if isinstance(conclusion, str) else str(conclusion))
        rules = [horn_rule(a) for a in asts]
        q = _conj(goal)
        if all(r is not None for r in rules) and q is not None:
            facts = [h for b, h in rules if not b and not any(is_var(x) for x in h[1][1])]
            rls = [r for r in rules if r[0] or any(is_var(x) for x in r[1][1][1])]
            dom = tuple(constants(facts, rls, q))
            c = (globals().get('CWA_DEFAULT', True) if cwa is None else cwa)
            kb = forward_chain(facts, rls, dom, c)
            ans = all(holds(l, kb, dom, c) for l in q)
            return (ans, 'logic-forward-chaining') if return_route else ans
    except Exception:
        pass
    ans = fol_engine(prem, conclusion)
    return (ans, 'logic-resolution') if return_route else ans

for f_ in ["∀x (Young(x) ∧ ¬Big(x) → Smart(x))", "Kind(bob)", "∀x (Tall(x) ∨ Short(x))"]:
    print(f"{f_:<36} -> Horn rule: {horn_rule(parse_fol_robust(f_)) is not None}")

∀x (Young(x) ∧ ¬Big(x) → Smart(x))   -> Horn rule: True
Kind(bob)                            -> Horn rule: True
∀x (Tall(x) ∨ Short(x))              -> Horn rule: False


In [27]:
def theory_to_logic(context):
    """NL theory -> logic strings (used to test the 'logic given directly' setting on public data)."""
    facts, rules, _ = parse_theory(context)
    def a2s(a): return f"{a[0].capitalize()}({', '.join(x[1:] if is_var(x) else x for x in a[1])})"
    def l2s(l): return ('¬' if l[0] else '') + a2s(l[1])
    out = [l2s(l) for l in facts]
    for body, head in rules:
        vs = sorted({x[1:] for _, a in list(body) + [head] for x in a[1] if is_var(x)})
        q = ''.join(f'∀{v} ' for v in vs)
        out.append(q + (f"({' ∧ '.join(l2s(l) for l in body)} → {l2s(head)})" if body else l2s(head)))
    return out


def statement_to_logic(statement):
    q = parse_query(statement)
    def a2s(a): return f"{a[0].capitalize()}({', '.join(x[1:] if is_var(x) else x for x in a[1])})"
    return ' ∧ '.join(('¬' if l[0] else '') + a2s(l[1]) for l in q) if q else None


print("logic form of the demo theory:")
for line in theory_to_logic(demo_theory): print("  ", line)
print("query:", statement_to_logic("Bob is successful."))

logic form of the demo theory:
   Young(bob)
   Kind(bob)
   ∀x (Young(x) → Smart(x))
   ∀x (Smart(x) ∧ Kind(x) → Successful(x))
query: Successful(bob)


In [28]:
_FOL_KB_CACHE = {}

def fol_engine(premises_fol, conclusion_fol, timeout=1.0, return_proof=False):
    premises_fol = split_formulas(premises_fol)
    key = tuple(premises_fol)
    ctr = itertools.count()
    if key not in _FOL_KB_CACHE:
        kb = []
        for p in premises_fol:
            try: kb += fol_to_cnf(parse_fol_robust(p), ctr)
            except Exception: pass              # skip an unparsable premise, keep the rest
        _FOL_KB_CACHE[key] = kb
    kb = _FOL_KB_CACHE[key]
    goal = fol_to_cnf(('not', parse_fol_robust(conclusion_fol)), itertools.count(10**6))
    res, parents = resolution_prove(kb, goal, timeout=timeout)
    ans = bool(res)
    return (ans, parents) if return_proof else ans


folio_demo = ["∀x (Dog(x) → Mammal(x))", "∀x (Mammal(x) → Animal(x))", "Dog(max)", "Tall(max) ⊕ Short(max)", "Short(max)"]
for c_ in ["Animal(max)", "¬Tall(max)", "Tall(max)"]:
    print(f"{c_:<12} -> resolution: {fol_engine(folio_demo, c_)}   | logic router: {logic_engine(folio_demo, c_, return_route=True)}")
print("RuleTaker-style logic:", logic_engine(theory_to_logic(demo_theory), statement_to_logic("Bob is successful."), return_route=True))

Animal(max)  -> resolution: True   | logic router: (True, 'logic-resolution')
¬Tall(max)   -> resolution: True   | logic router: (True, 'logic-resolution')
Tall(max)    -> resolution: False   | logic router: (False, 'logic-resolution')
RuleTaker-style logic: (True, 'logic-forward-chaining')


## 6. Router — one entry point for any (context, statement)
Chooses the representation automatically: FOL formulas available → resolution; causal language → SCM; otherwise → NL Horn-rule parser + forward chaining.
If nothing can be parsed, the system falls back to **False** (not entailed), which is the correct default for an entailment checker.

In [29]:
CWA_DEFAULT = True          # chosen from the public-data experiment in section 8

def looks_causal(text):
    t = text.lower()
    return bool(re.search(r'\b(causes?|leads? to)\b', t)) and bool(re.search(r'\bwould\b', t))


def looks_fol(s):
    return bool(s) and bool(re.search(r'[∀∃¬∧∨→↔⊕]|\w+\([\w, ]+\)', s))


def predict(context, statement, premises_fol=None, conclusion_fol=None, cwa=None):
    """Returns (bool prediction, engine name)."""
    cwa = CWA_DEFAULT if cwa is None else cwa
    try:
        if premises_fol and conclusion_fol:
            return logic_engine(premises_fol, conclusion_fol, cwa=cwa, return_route=True)
        if looks_fol(statement) and looks_fol(context) and not looks_causal(context + statement):
            return logic_engine(context, statement, cwa=cwa, return_route=True)
        if looks_causal(context + ' ' + statement):
            r = causal_engine(context, statement)
            if r is not None: return r, 'scm'
        r = rule_engine(context, statement, cwa=cwa)
        if r is not None: return r, 'forward-chaining'
    except Exception as e:
        pass
    return False, 'fallback'

for ctx_, q_ in [(demo_theory, "Bob is successful."), (demo_cb, "Would Zlim occur if not Glent instead of Glent?"), ("\n".join(folio_demo), "Animal(max)")]:
    print(f"{q_:<50} -> {predict(ctx_, q_)}")

Bob is successful.                                 -> (True, 'forward-chaining')
Would Zlim occur if not Glent instead of Glent?    -> (False, 'scm')
Animal(max)                                        -> (True, 'logic-resolution')


## 7. Data loading (RuleTaker, FOLIO, CounterBench from Hugging Face)
Field names differ between datasets, so every row is mapped to a common schema
`{id, context, statement, premises_fol, conclusion_fol, label}`.

In [30]:
TRUE_STR = {'true', 'entailment', 'entailed', 'yes', '1', 'proved', 'valid'}

def to_bool(v):
    if isinstance(v, bool): return v
    return str(v).strip().lower() in TRUE_STR


def to_example(row, label_names=None, idx=None):
    low = {k.lower().replace('_', '-'): k for k in row}
    def get(*names):
        for n in names:
            k = low.get(n)
            if k is not None and row[k] not in (None, '', []): return row[k]
        return None
    ctx = get('context', 'theory', 'premises', 'story', 'facts', 'kb', 'passage', 'given-info', 'prompt')
    st = get('statement', 'question', 'query', 'hypothesis', 'conclusion', 'text')
    pf = get('premises-fol', 'fol-premises', 'premises-fol-str')
    cf = get('conclusion-fol', 'fol-conclusion')
    lab = get('label', 'answer', 'flag', 'gold', 'target', 'gold-label')
    if isinstance(ctx, list): ctx = '\n'.join(ctx)
    if isinstance(pf, list): pf = '\n'.join(pf)
    if isinstance(lab, int) and label_names: lab = label_names[lab]
    return {'id': get('id', 'example-id', 'story-id') if get('id', 'example-id') is not None else idx,
            'context': ctx or '', 'statement': st or '', 'premises_fol': pf, 'conclusion_fol': cf,
            'label': None if lab is None else to_bool(lab), 'raw_label': lab,
            'config': get('config', 'depth', 'qdep', 'type', 'category')}


def load_hf(name, split, config=None, limit=None):
    from datasets import load_dataset
    ds = load_dataset(name, config, split=split) if config else load_dataset(name, split=split)
    names = None
    for k in ('label', 'answer'):
        if k in ds.features and hasattr(ds.features[k], 'names'): names = ds.features[k].names
    rows = ds if limit is None else ds.shuffle(seed=42).select(range(min(limit, len(ds))))   # random sample covers all depths/configs
    return [to_example(r, names, i) for i, r in enumerate(rows)]

print(to_example({'context': 'Bob is red.', 'question': 'Bob is red.', 'label': 'entailment', 'config': 'depth-0'}))

{'id': None, 'context': 'Bob is red.', 'statement': 'Bob is red.', 'premises_fol': None, 'conclusion_fol': None, 'label': True, 'raw_label': 'entailment', 'config': 'depth-0'}


In [31]:
FOLIO_GITHUB = 'https://raw.githubusercontent.com/Yale-LILY/FOLIO/main/data/v0.0/folio-{split}.jsonl'

def load_url_jsonl(url, limit=None):
    """Load a public JSONL file directly (no Hugging Face login needed)."""
    import urllib.request
    lines = urllib.request.urlopen(url, timeout=60).read().decode('utf-8').splitlines()
    rows = [json.loads(l) for l in lines if l.strip()][:limit]
    return [to_example(r, idx=i) for i, r in enumerate(rows)]


def load_local(path, limit=None):
    """Load a dataset file you downloaded yourself (csv / json / jsonl)."""
    rows = read_any(path)[:limit]
    return [to_example(r, idx=i) for i, r in enumerate(rows)]

COUNTERBENCH_FILES = ['data_balanced_alpha_V1.json', 'data_balanced_backdoor_V2.json']
COUNTERBENCH_META = {'data_balanced_alpha_V1.json': 'meta_model_alpha_V1.json',
                     'data_balanced_backdoor_V2.json': 'meta_model_backdoor_V2.json'}

def load_counterbench(repo='CounterBench/CounterBench', limit=None):
    """The HF repo mixes data and meta-model files with different columns, so `load_dataset` fails.
    We download the data files directly and attach each causal model's background text (via model_id)."""
    from huggingface_hub import hf_hub_download
    out = []
    for fn in COUNTERBENCH_FILES:
        rows = json.load(open(hf_hub_download(repo, fn, repo_type='dataset')))
        try:
            meta = json.load(open(hf_hub_download(repo, COUNTERBENCH_META[fn], repo_type='dataset')))
            bg = {m.get('model_id'): m.get('background', '') for m in meta}
        except Exception:
            bg = {}
        for r in rows:
            mid = (r.get('meta') or {}).get('model_id')
            ex = to_example(r, idx=len(out))
            ex['context'] = (bg.get(mid, '') + ' ' + ex['context']).strip()
            ex['config'] = r.get('type') or (r.get('meta') or {}).get('query_type')
            out.append(ex)
    random.Random(42).shuffle(out)
    return out[:limit] if limit else out

print("loaders ready: load_hf, load_url_jsonl, load_local, load_counterbench")

loaders ready: load_hf, load_url_jsonl, load_local, load_counterbench


## 8. Evaluation: accuracy + total inference time

In [32]:
def evaluate(examples, name='', cwa=None, verbose=True):
    t0 = time.perf_counter()
    correct, engines, by_cfg = 0, Counter(), defaultdict(lambda: [0, 0])
    preds = []
    for ex in examples:
        p, eng = predict(ex['context'], ex['statement'], ex['premises_fol'], ex['conclusion_fol'], cwa=cwa)
        preds.append(p); engines[eng] += 1
        ok = (p == ex['label'])
        correct += ok
        c = by_cfg[str(ex.get('config'))]; c[0] += ok; c[1] += 1
    dt = time.perf_counter() - t0
    acc = 100 * correct / max(1, len(examples))
    if verbose:
        print(f"[{name}] n={len(examples)}  accuracy={acc:.2f}%  total_time={dt:.3f}s  "
              f"({1000*dt/max(1,len(examples)):.3f} ms/query)  engines={dict(engines)}")
        if len(by_cfg) > 1:
            for k, (a, n) in sorted(by_cfg.items()): print(f"     {k:>20}: {100*a/n:6.2f}%  (n={n})")
    return acc, dt, preds


_toy = [{'context': demo_theory, 'statement': s_, 'premises_fol': None, 'conclusion_fol': None, 'label': l_, 'config': 'toy'}
        for s_, l_ in [("Bob is successful.", True), ("Bob is big.", False), ("Bob is not big.", True)]]
evaluate(_toy, 'toy check')

[toy check] n=3  accuracy=100.00%  total_time=0.000s  (0.087 ms/query)  engines={'forward-chaining': 3}


(100.0, 0.0002611130475997925, [True, False, True])

## 9. Submission generation
The private test file is expected as CSV/JSON/JSONL with an `id` column plus context/statement (FOL columns optional).

In [33]:
def read_any(path):
    if path.endswith('.csv'):
        import pandas as pd
        return pd.read_csv(path).fillna('').to_dict('records')
    if path.endswith('.jsonl'):
        return [json.loads(l) for l in open(path) if l.strip()]
    return json.load(open(path))


def make_submission(test_path, out_path='submission.csv', label_col='label'):
    rows = read_any(test_path)
    t0 = time.perf_counter()
    with open(out_path, 'w', newline='') as f:
        w = csv.writer(f); w.writerow(['id', label_col])
        for i, r in enumerate(rows):
            ex = to_example(r, idx=i)
            p, _ = predict(ex['context'], ex['statement'], ex['premises_fol'], ex['conclusion_fol'])
            w.writerow([ex['id'] if ex['id'] is not None else i, 'True' if p else 'False'])
    print(f"wrote {out_path}: {len(rows)} predictions in {time.perf_counter()-t0:.3f}s")

print("submission writer ready: make_submission(test_path, 'submission.csv')")

submission writer ready: make_submission(test_path, 'submission.csv')


## 10. Demo on the example from the assignment (with reasoning trace)

In [34]:
demo_ctx = ("Bob is young. Bob is kind. If someone is young, then they are smart. "
            "If someone is smart and kind, then they are successful.")
ans, kb, q = rule_engine(demo_ctx, "Bob is successful.", return_kb=True)
print("Parsed facts :", [fmt_lit(l) for l in parse_theory(demo_ctx)[0]])
print("Parsed rules :", [fmt_rule(r) for r in parse_theory(demo_ctx)[1]])
print("Prediction   :", ans)
print("\n".join(explain(q[0], kb)))

Parsed facts : ['young(bob)', 'kind(bob)']
Parsed rules : ['young(X) → smart(X)', 'smart(X) ∧ kind(X) → successful(X)']
Prediction   : True
successful(bob)   [rule: smart(X) ∧ kind(X) → successful(X)  θ={X/bob}]
    smart(bob)   [rule: young(X) → smart(X)  θ={X/bob}]
        young(bob)   [fact]
    kind(bob)   [fact]


## 11. Load public datasets
* No Hugging Face login needed: RuleTaker / CounterBench are public on HF; FOLIO is read from the authors' public GitHub copy.
* Set `COUNTERBENCH_ID` to the exact Hugging Face id used in the course (or put a downloaded file path in `COUNTERBENCH_FILE`).

In [35]:
import sys, subprocess, importlib.util
if importlib.util.find_spec('datasets') is None:      # install into the *current* kernel / interpreter
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'datasets', 'pandas'], check=False)
RULETAKER_ID, FOLIO_ID = 'tasksource/ruletaker', 'yale-nlp/FOLIO'
COUNTERBENCH_ID = 'CounterBench/CounterBench'
COUNTERBENCH_FILE = None      # e.g. 'counterbench.json' if you download it manually
LIMIT = None          # per dataset, for speed; set None for full split

def try_load(name, limit=LIMIT, config=None):
    for split in ('test', 'validation', 'dev', 'train'):
        try:
            d = load_hf(name, split, config, limit)
            print(f"loaded {name}:{split}  n={len(d)}"); return d
        except Exception as e:
            err = e
    print(f"could not load {name}: {err}"); return []

def load_folio(limit=LIMIT):
    try:
        d = load_url_jsonl(FOLIO_GITHUB.format(split='validation'), limit)
        print(f"loaded FOLIO (GitHub) validation  n={len(d)}"); return d
    except Exception as e:
        print("FOLIO GitHub download failed, trying Hugging Face:", e); return try_load(FOLIO_ID)

def safe_cb():
    try:
        d = load_local(COUNTERBENCH_FILE) if COUNTERBENCH_FILE else load_counterbench(COUNTERBENCH_ID, LIMIT)
        print(f"loaded CounterBench  n={len(d)}"); return d
    except Exception as e:
        print("could not load CounterBench:", e); return []

data = {'RuleTaker': try_load(RULETAKER_ID),
        'FOLIO': load_folio(),
        'CounterBench': safe_cb()}
for k, v in data.items():
    if v: print(k, 'example:', {kk: (str(vv)[:120]) for kk, vv in v[0].items()})

loaded tasksource/ruletaker:test  n=151911


loaded FOLIO (GitHub) validation  n=204


loaded CounterBench  n=1200
RuleTaker example: {'id': '0', 'context': 'Anne is big. Anne is cold. Anne is furry. Anne is green. Anne is red. Anne is rough. Erin is cold. Erin is furry. Erin i', 'statement': 'Anne is red.', 'premises_fol': 'None', 'conclusion_fol': 'None', 'label': 'True', 'raw_label': 'entailment', 'config': 'depth-1'}
FOLIO example: {'id': '0', 'context': 'If people perform in school talent shows often, then they attend and are very engaged with school events.\nPeople either ', 'statement': 'Bonnie performs in school talent shows often.', 'premises_fol': '∀x (TalentShows(x) → Engaged(x))\n∀x (TalentShows(x) ∨ Inactive(x))\n∀x (Chaperone(x) → ¬Students(x))\n∀x (Inactive(x) → Ch', 'conclusion_fol': 'Engaged(bonnie)', 'label': 'False', 'raw_label': 'Uncertain', 'config': 'None'}
CounterBench example: {'id': '439', 'context': 'Imagine a self-contained, hypothetical world with only the following conditions, and without any unmentioned factors or ', 'statement': 'Would Wrox

## 12. Experiment: closed-world vs open-world semantics (chosen on public data only)

In [36]:
if data['RuleTaker']:
    acc_cwa, _, _ = evaluate(data['RuleTaker'], 'RuleTaker / CWA (NAF)', cwa=True)
    acc_owa, _, _ = evaluate(data['RuleTaker'], 'RuleTaker / OWA', cwa=False)
    CWA_DEFAULT = acc_cwa >= acc_owa
    print("=> using", "CWA" if CWA_DEFAULT else "OWA")

# CounterBench: reading conventions of counterfactual questions, tuned per question type
# on a 50% selection split and evaluated on the untouched 50% held-out split.
if data['CounterBench']:
    print("Tuning SCM reading conventions (selection split):")
    cb_sel, cb_held = tune_scm(data['CounterBench'])
    saved = SCM_CFG; SCM_CFG = {}
    evaluate(cb_held, 'CounterBench held-out / default reading')
    SCM_CFG = saved
    evaluate(cb_held, 'CounterBench held-out / tuned reading')

[RuleTaker / CWA (NAF)] n=151911  accuracy=99.97%  total_time=13.472s  (0.089 ms/query)  engines={'forward-chaining': 151911}
                  NatLang: 100.00%  (n=7793)
                  depth-0: 100.00%  (n=17347)
                  depth-1: 100.00%  (n=17788)
                  depth-2:  99.99%  (n=17741)
                  depth-3:  99.96%  (n=17982)
               depth-3ext:  99.97%  (n=23222)
       depth-3ext-NatLang:  99.97%  (n=31017)
                  depth-5:  99.92%  (n=19021)


[RuleTaker / OWA] n=151911  accuracy=74.57%  total_time=7.317s  (0.048 ms/query)  engines={'forward-chaining': 151911}
                  NatLang:  75.67%  (n=7793)
                  depth-0:  77.94%  (n=17347)
                  depth-1:  74.15%  (n=17788)
                  depth-2:  73.61%  (n=17741)
                  depth-3:  73.46%  (n=17982)
               depth-3ext:  73.71%  (n=23222)
       depth-3ext-NatLang:  74.23%  (n=31017)
                  depth-5:  74.97%  (n=19021)
=> using CWA
Tuning SCM reading conventions (selection split):


         joint: default  92.98%  ->  chosen  92.98%  (default kept)  cfg={'neg_obs': False, 'neg_do': False, 'clamp': False, 'incons': False, 'multi': 'all'}


        nested: default  96.18%  ->  chosen  96.18%  (default kept)  cfg={'neg_obs': False, 'neg_do': False, 'clamp': False, 'incons': False, 'multi': 'all'}


      backdoor: default  86.79%  ->  chosen  91.51%  (switched)  cfg={'neg_obs': True, 'neg_do': False, 'clamp': False, 'incons': False, 'multi': 'all'}


   conditional: default  74.38%  ->  chosen  93.39%  (switched)  cfg={'neg_obs': True, 'neg_do': False, 'clamp': False, 'incons': False, 'multi': 'all'}


         basic: default 100.00%  ->  chosen 100.00%  (default kept)  cfg={'neg_obs': False, 'neg_do': False, 'clamp': False, 'incons': False, 'multi': 'all'}
[CounterBench held-out / default reading] n=600  accuracy=90.50%  total_time=0.121s  (0.202 ms/query)  engines={'scm': 600}
                 backdoor:  93.62%  (n=94)
                    basic: 100.00%  (n=122)
              conditional:  65.89%  (n=129)
                    joint:  98.53%  (n=136)
                   nested:  95.80%  (n=119)


[CounterBench held-out / tuned reading] n=600  accuracy=94.17%  total_time=0.123s  (0.205 ms/query)  engines={'scm': 600}
                 backdoor:  93.62%  (n=94)
                    basic: 100.00%  (n=122)
              conditional:  82.95%  (n=129)
                    joint:  98.53%  (n=136)
                   nested:  95.80%  (n=119)


## 13. Final accuracy and total inference time

In [37]:
_THEORY_CACHE.clear(); _FC_CACHE.clear(); _FOL_KB_CACHE.clear()   # honest timing: cold caches
all_ex, total_t = [], 0.0
for name, exs in data.items():
    if not exs: continue
    acc, dt, _ = evaluate(exs, name)
    all_ex += exs; total_t += dt
if all_ex:
    acc, dt, preds = evaluate(all_ex, 'ALL', verbose=False)
    print(f"\nFINAL ACCURACY = {acc:.2f}%   |   TOTAL INFERENCE TIME = {total_t:.3f} s for {len(all_ex)} queries")

[RuleTaker] n=151911  accuracy=99.97%  total_time=11.305s  (0.074 ms/query)  engines={'forward-chaining': 151911}
                  NatLang: 100.00%  (n=7793)
                  depth-0: 100.00%  (n=17347)
                  depth-1: 100.00%  (n=17788)
                  depth-2:  99.99%  (n=17741)
                  depth-3:  99.96%  (n=17982)
               depth-3ext:  99.97%  (n=23222)
       depth-3ext-NatLang:  99.97%  (n=31017)
                  depth-5:  99.92%  (n=19021)


[FOLIO] n=204  accuracy=98.04%  total_time=3.925s  (19.240 ms/query)  engines={'logic-resolution': 190, 'logic-forward-chaining': 14}


[CounterBench] n=1200  accuracy=94.58%  total_time=0.246s  (0.205 ms/query)  engines={'scm': 1200}
                 backdoor:  92.50%  (n=200)
                    basic: 100.00%  (n=250)
              conditional:  88.00%  (n=250)
                    joint:  96.00%  (n=250)
                   nested:  96.00%  (n=250)



FINAL ACCURACY = 99.93%   |   TOTAL INFERENCE TIME = 15.476 s for 153315 queries


## 13b. Setting where the test gives the **logic directly**
Each RuleTaker theory is translated to logic formulas (e.g. `∀x (Young(x) ∧ ¬Big(x) → Smart(x))`) and fed
through the logic route of the router, to check that logic input gives the same answers as NL input.

In [38]:
if data['RuleTaker']:
    logic_ex = []
    for ex in data['RuleTaker']:
        q = statement_to_logic(ex['statement'])
        if q: logic_ex.append({**ex, 'context': '\n'.join(theory_to_logic(ex['context'])), 'statement': q})
    print("example logic KB:\n  " + '\n  '.join(logic_ex[0]['context'].split('\n')[:6]), "\n  query:", logic_ex[0]['statement'])
    evaluate(logic_ex, 'RuleTaker given as logic')

example logic KB:
  Big(anne)
  Cold(anne)
  Furry(anne)
  Green(anne)
  Red(anne)
  Rough(anne) 
  query: Red(anne)


[RuleTaker given as logic] n=151911  accuracy=99.97%  total_time=110.393s  (0.727 ms/query)  engines={'logic-forward-chaining': 151911}
                  NatLang: 100.00%  (n=7793)
                  depth-0: 100.00%  (n=17347)
                  depth-1: 100.00%  (n=17788)
                  depth-2:  99.99%  (n=17741)
                  depth-3:  99.96%  (n=17982)
               depth-3ext:  99.97%  (n=23222)
       depth-3ext-NatLang:  99.97%  (n=31017)
                  depth-5:  99.92%  (n=19021)


## 14. Error analysis — sentences the parser could not translate (guides parser improvements)

In [39]:
unparsed = Counter()
for ex in data['RuleTaker'] + data['CounterBench']:
    if looks_causal(ex['context'] + ex['statement']): continue
    for s in parse_theory(ex['context'])[2]: unparsed[normalize(s)] += 1
print("distinct unparsed sentences:", len(unparsed))
for s, c in unparsed.most_common(15): print(f"{c:5d}  {s}")

# Wrong predictions per dataset (first few) - used to improve the parsers
for name, exs in data.items():
    wrong = [ex for ex in exs if predict(ex['context'], ex['statement'], ex['premises_fol'], ex['conclusion_fol'])[0] != ex['label']]
    print(f"\n{name}: {len(wrong)} wrong")
    if name == 'CounterBench':
        by_t = defaultdict(list)
        for ex in wrong: by_t[ex.get('config')].append(ex)
        for t, lst in by_t.items():
            for ex in lst[:2]:
                gi = ex['context'].split('We know that')[-1]
                print(f"  [{t}] We know that{gi[:400]} | Q: {ex['statement']} | gold: {ex['raw_label']}")
                res = causal_engine(ex['context'], ex['statement'], return_info=True)
                print(fmt_scm(res[1]) if res else "    (SCM parse failed)")
        continue
    for ex in wrong[:4]:
        print("  [", ex.get('config'), "]", ex['context'][-300:], "| Q:", ex['statement'], "| gold:", ex['raw_label'])

distinct unparsed sentences: 15
    4  blen~bern(0.4)
    4  plim~bern(0.4)
    4  razz~bern(0.6)
    4  trune~bern(0.4)
    4  vank~bern(0.6)
    4  blorn~bern(0.6)
    4  blaf~bern(0.6)
    4  pex~bern(0.4)
    4  scud~bern(0.4)
    4  splee~bern(0.6)
    2  blorn~bern(0.4)
    2  splee~bern(0.4)
    2  blaf~bern(0.4)
    2  razz~bern(0.4)
    2  vank~bern(0.4)



RuleTaker: 41 wrong
  [ depth-5 ] n. If the bear does not see the cow then the bear eats the rabbit. If someone sees the rabbit and they do not eat the lion then the lion sees the rabbit. If the rabbit eats the lion and the rabbit is not green then the rabbit sees the bear. If someone does not see the rabbit then they like the bear. | Q: The bear does not like the bear. | gold: entailment
  [ depth-5 ] n. If the bear does not see the cow then the bear eats the rabbit. If someone sees the rabbit and they do not eat the lion then the lion sees the rabbit. If the rabbit eats the lion and the rabbit is not green then the rabbit sees the bear. If someone does not see the rabbit then they like the bear. | Q: The lion likes the bear. | gold: not entailment
  [ depth-5 ]  the dog likes the cow. If someone eats the dog and they are rough then the dog eats the cow. All big people are nice. If someone visits the dog and the dog is rough then the dog is big. If someone does not visit the lion the


FOLIO: 4 wrong
  [ None ] ome.
If an employee has lunch at home, then he/she is working remotely from home.
All employees who are in other countries work remotely from home. 
No managers work remotely from home. 
James is either a manager and appears in the company today or neither a manager nor appears in the company today. | Q: If James is either a manager or in other countries,  then James either has lunch at home and works remotely from home, or neither has lunch at home nor works remotely from home. | gold: True
  [ None ] uperhero movie is named after a character, that character is a good guy.
The Surprising Adventures of Sir Digby Chicken Caesar is named after Sir Digby.
If somebody wins a fight, the person they are fighting loses.
If a superhero movie is named after a character, that character appears in the movie. | Q: Sir Digby’s nemesis loses. | gold: True
  [ None ] The Picuris Mountains are a mountain range in New Mexico or Texas.
Juan de Onate visited the Picuris Mountai


CounterBench: 65 wrong
  [joint] We know that Fizo causes Blorn, Blorn causes Plim, Plim causes not Quaz, Quaz or Blorn causes Skul, Skul causes Triv, and Triv causes Jext. | Q: Would Jext occur if not Fizo and not Quaz? | gold: yes
  blorn := (fizo)
  plim := (blorn)
  quaz := ¬(plim)
  skul := (quaz ∨ blorn)
  triv := (skul)
  jext := (triv)
  type: joint   evidence/factual: {'fizo': True}   do: {'fizo': False, 'quaz': False}
  [joint] We know that Ziklo causes not Blaf, Blaf causes Trune, Trune and Blaf together cause Vork, and Vork causes Lumbo. | Q: Would Lumbo occur if not Ziklo and not Vork? | gold: yes
  blaf := ¬(ziklo)
  trune := (blaf)
  vork := (trune ∧ blaf)
  lumbo := (vork)
  type: joint   evidence/factual: {'ziklo': True}   do: {'ziklo': False, 'vork': False}
  [conditional] We know that Ziklo causes not Blaf, Blaf causes Trune, Trune and not Blaf together cause Vork, Vork causes Sline, Frim, and Qado, Frim or not Qado causes Jurf, and Jurf causes Lumbo. We observed Vo

## 15. Reasoning traces (used in the report)

In [40]:
cands = []
for ex in data['RuleTaker'][:3000]:
    r = rule_engine(ex['context'], ex['statement'], cwa=CWA_DEFAULT, return_kb=True)
    if r and r[0] and r[0] == ex['label'] and len(r[2]) == 1:
        lines = explain(r[2][0], r[1], CWA_DEFAULT)
        cands.append((len(lines), ex, lines))
cands.sort(key=lambda t: -t[0])          # show the deepest proofs (most informative for the report)
for _, ex, lines in cands[:2]:
    print("Q:", ex['statement'], "| config:", ex['config'], "| gold:", ex['label'])
    print("\n".join(lines)); print('-' * 80)
for ex in data['CounterBench']:
    if ex.get('config') in ('nested', 'backdoor'):
        res = causal_engine(ex['context'], ex['statement'], return_info=True)
        if res is None: continue
        a, info = res
        if a == ex['label']:
            print("CounterBench [", ex['config'], "]", ex['statement'], "| gold:", ex['raw_label'], "| pred:", a)
            print(fmt_scm(info)); print('-' * 80); break
for ex in data['FOLIO']:
    ok, par = fol_engine(ex['premises_fol'], ex['conclusion_fol'], return_proof=True)
    if ok:
        print("Conclusion:", ex['conclusion_fol'], "| gold:", ex['label'])
        print("\n".join(proof_lines(par))); break

Q: Fiona is not nice. | config: depth-1 | gold: True
¬nice(fiona)   [rule: ¬round(X) → ¬nice(X)  θ={X/fiona}]
    ¬round(fiona)   [rule: blue(X) ∧ ¬red(X) → ¬round(X)  θ={X/fiona}]
        blue(fiona)   [fact]
        ¬red(fiona)   [negation-as-failure: red(fiona) not derivable]
--------------------------------------------------------------------------------
Q: Erin is young. | config: depth-1 | gold: True
young(erin)   [rule: quiet(X) ∧ white(X) → young(X)  θ={X/erin}]
    quiet(erin)   [fact]
    white(erin)   [rule: quiet(X) → white(X)  θ={X/erin}]
        quiet(erin)   [fact]
--------------------------------------------------------------------------------
CounterBench [ nested ] Assume not Glent, and based on this assumption, further suppose not Melf. Would Zlim occur? | gold: no | pred: False
  razz := (glent)
  pex := (razz)
  zurn := (pex)
  melf := (zurn)
  zlim := (melf)
  type: nested   evidence/factual: {}   do: {'glent': False, 'melf': False}
-------------------------------

## 16. Generate `submission.csv`
Put the private test path in `TEST_PATH` (CSV / JSON / JSONL with `id`, context and statement columns).
If it is absent, a demo submission is produced from the public data so the notebook always runs end-to-end.

In [41]:
import os
TEST_PATH = 'test.csv'
if os.path.exists(TEST_PATH):
    make_submission(TEST_PATH, 'submission.csv')
else:
    t0 = time.perf_counter()
    with open('submission.csv', 'w', newline='') as f:
        w = csv.writer(f); w.writerow(['id', 'label'])
        for i, ex in enumerate(all_ex):
            p, _ = predict(ex['context'], ex['statement'], ex['premises_fol'], ex['conclusion_fol'])
            w.writerow([i, 'True' if p else 'False'])
    print(f"(demo) wrote submission.csv with {len(all_ex)} rows in {time.perf_counter()-t0:.3f}s")

(demo) wrote submission.csv with 153315 rows in 9.803s
